In [1]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import plotly.express as px
from flipside import Flipside
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import time
from memory_profiler import profile
import json
import csv
import os
import logging
import sys
from importlib import reload
from pymongo import MongoClient
from web3 import Web3
import psycopg2
from psycopg2.extras import execute_values
from io import StringIO
from typing import Any, Dict
from keys import KEYS

In [2]:
# logging configurations
reload(logging)
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

## Extract Data

In [4]:
# Initilize Flipside Client
flipside_key = KEYS['flipside_key']
flipside = Flipside(flipside_key, "https://api-v2.flipsidecrypto.xyz")

In [5]:
def format_query(query_path: str, params: dict) -> str:
    try:
        with open(query_path, "r") as file:
            query = file.read()
        formatted_query = query.format(**params)
        return formatted_query
    except Exception as e:
        raise ValueError(f"format query error: {e}")

In [6]:
def createQueryRun(query : str, api_key:str = flipside_key) -> str :
    
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }

    # Request payload
    payload = {
        "jsonrpc": "2.0",
        "method": "createQueryRun",
        "params": [
            {
                "resultTTLHours": 1,
                "maxAgeMinutes": 0,
                "sql": query ,
                "tags": {
                    "source": "postman-demo",
                    "env": "test"
                },
                "dataSource": "snowflake-default",
                "dataProvider": "flipside"
            }
        ],
        "id": 1
    }

    # Submit createQueryRun request
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        if response.status_code == 200:
            logging.info("Query run created successfully!")
            logging.debug(response.json())  # Output the response
            return  response.json()['result']['queryRequest']['queryRunId']
        else:
            raise requests.exceptions.HTTPError(
                f"Unexpected status code: {response.status_code}. Details: {response.text}" )
    except Exception as e:
        logging.error(f" createQueryRun Error: {e}")
    

In [7]:
def getQueryRun(queryRunId:str , api_key:str = flipside_key) -> str:
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }
    
    payload = {
    "jsonrpc": "2.0",
    "method": "getQueryRun",
    "params": [
        {
            "queryRunId": queryRunId
        }
    ],
    "id": 1
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        logging.debug(f'getQueryRun state {response.json()['result']['queryRun']['state']}')
        return response.json()['result']['queryRun']['state']
        
    except Exception as e:
        logging.error(f" getQueryRun Error: {e}")

In [8]:
def queryresult_Pagination(queryRunId:str, page_size:int = 70000) -> list:
       
    current_page_number = 1
    total_pages = 3

    all_rows = []

    while current_page_number <= total_pages:

        try:
            results = flipside.get_query_results(
                queryRunId,
                page_number=current_page_number,
                page_size=page_size         
            )
       
            if results.records:
                total_pages = results.page.totalPages
                all_rows.extend(results.records)
                logging.debug(f"Current page number: {current_page_number} Total Pages: {total_pages}, Rows Retrieved: {len(results.records)}")
            else: 
                logging.warning('No record')
                break

        except Exception as e:
            logging.error(f" Pagination Error: {e}")
            return None
        
        current_page_number += 1

    logging.info(f"Total Pages: {total_pages}, Rows Retrieved: {len(all_rows)}")
        
        
    return all_rows

In [9]:
def extract_flipsidecrypto_data(query_path:str, params: dict , api_key = flipside_key, retry_time:int = 90 ,timeout:int = 600 ) -> list:
    
    try:
        logging.info(f'Start query with params:{params}')
        query = format_query(query_path,params)
        queryRunId = createQueryRun(query,api_key)

        state = None
        start_time = time.time()

        while state != 'QUERY_STATE_SUCCESS':
            
            state = getQueryRun(queryRunId,api_key)

            if state == 'QUERY_STATE_SUCCESS':
                 break 

            elif state in ['QUERY_STATE_FAILED', 'QUERY_STATE_CANCELED']:
                raise RuntimeError(f"Query execution failed or was canceled. State: {state}")
            
            elif state in ['QUERY_STATE_STREAMING_RESULTS', 'QUERY_STATE_RUNNING', 'QUERY_STATE_READY']:
                if time.time() - start_time > timeout:
                    raise TimeoutError("Query execution exceeded timeout limit.")
                
                logging.info(f"Wainting query excution")
                logging.debug(f"retry after {retry_time} sec")

                time.sleep(retry_time)

            else: raise ValueError(f"Unexpected query state: {state}")

            
        result = queryresult_Pagination(queryRunId)

    except TimeoutError as e:
        logging.error(f"Timeout Error: {e}")
        return None
    except RuntimeError as e:
        logging.error(f"Runtime Error: {e}")
        return None
    except Exception as e:
        logging.error(f" state Error: {e}")
        return None
    
                   

    return result

In [10]:
pool_address = '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'

In [11]:
positin_data_query_path = r'sql_queries\get_position_data_query.sql'
positions_extracted_data = extract_flipsidecrypto_data(positin_data_query_path,params= {'pool_address': pool_address})

2024-10-06 16:54:07 - INFO - Start query with params:{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'}
2024-10-06 16:54:08 - INFO - Query run created successfully!
2024-10-06 16:54:08 - INFO - Wainting query excution
2024-10-06 16:55:39 - INFO - Wainting query excution
2024-10-06 16:57:09 - INFO - Wainting query excution
2024-10-06 16:58:40 - INFO - Wainting query excution
2024-10-06 17:04:51 - INFO - Total Pages: 7, Rows Retrieved: 453982


In [12]:
# positions_extracted_data -> list[dict{}] 
positions_extracted_data[1]

{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'block_hash': '0xa084e5834ce2c9c44b7f00a29f98221c5f1bd0581a1464d46e03908f874e6e18',
 'block_number': 12389533,
 'tx_hash': '0xc5feb5cd2c29c98cd578ce38c9392c457777f39198fae383e093d56aa28c51ba',
 'tx_index': 56,
 'contract_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'event_index': 75,
 'block_timestamp': '2021-05-07T21:11:46.000Z',
 'origin_from_address': '0x6a5d7be330f024771ab04a836b5e41406b7a2757',
 'origin_to_address': '0xc36442b4a4522e871399cd717abdd847ab11fe88',
 'topic0': '0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde',
 'event_name': 'Mint',
 'topics': ['0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde',
  '0x000000000000000000000000c36442b4a4522e871399cd717abdd847ab11fe88',
  '0x000000000000000000000000000000000000000000000000000000000002f8dc',
  '0x000000000000000000000000000000000000000000000000000000000002f8e6'],
 'data': '0x000000000000000000000000c36442b4a4522e

In [13]:
pool_info_query_path = r'sql_queries\get_pool_info_query.sql'
pool_info_data = extract_flipsidecrypto_data(pool_info_query_path, params= {'pool_address': pool_address} )

2024-10-06 17:04:51 - INFO - Start query with params:{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'}
2024-10-06 17:04:51 - INFO - Query run created successfully!
2024-10-06 17:04:52 - INFO - Wainting query excution
2024-10-06 17:06:23 - INFO - Total Pages: 1, Rows Retrieved: 1


In [14]:
pool_info_data[0]

{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'token0': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'token1': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'fee': 500,
 'tickspacing': 10,
 '__row_index': 0}